# Dynamic sparse JWST AMI calibration and binary inference

This tutorial builds a genuinely sparse dynamic JWST/NIRISS aperture-masking model, calibrates its geometric distortion from a reference star, exports the recovered model as a static sparse optic, and then uses the calibrated instrument for Hamiltonian Monte Carlo inference of a close binary. The two-stage workflow separates instrument calibration from the more expensive astrophysical posterior calculation.

The tutorial has four stages:

1. **Build a dynamic sparse aperture.** Start from the nominal seven-hole `JWSTNRMLike` geometry and construct a `SparseDynamicOptic` that never materialises the complete pupil. Parameterise the departure from the ideal aperture with a polynomial `DistortCoords` transformation, retaining explicit physical `(x, y)` hole centres and local sparse pupil sampling. Inspect the nominal and distorted hole geometry and verify the leading aperture-axis contract.
2. **Recover the aperture distortion.** Inject known polynomial coefficients, simulate a noisy unresolved reference-star exposure, reset the calibration model, and recover the distortion coefficients with a differentiable image objective. Show parameter histories, reference/model/z-score residuals, and injected-versus-recovered aperture geometry. The calibration should solve the polynomial distortion itself rather than fitting seven unrelated hole positions.
3. **Freeze the calibrated instrument.** Evaluate the recovered polynomial mapping once and export an equivalent static `SparseOptic`. Confirm that the dynamic recovered model and static sparse model agree in their hole locations, complex field, and focal-plane PSF to numerical tolerance. This removes dynamic coordinate generation from every later likelihood evaluation and makes the calibrated aperture a compact reusable instrument model.
4. **Infer a close binary with HMC.** Hold the static sparse optic fixed, construct a physical `BinarySource`, and infer separation, position angle, contrast, total flux, and any necessary centring parameter with Hamiltonian Monte Carlo using NumPyro or BlackJAX. Initialise from a deterministic fit, run and diagnose the chains, compare injected and posterior-recovered parameters, and finish with posterior predictive model/data/z-score images.

The central recipe is therefore `dynamic sparse optic → polynomial aperture calibration → static sparse optic → binary-source HMC`. Dense aperture materialisation, spectral-information modes, and simultaneous instrument-and-binary sampling are deliberately outside this tutorial. Timing comparisons must separate compilation from steady-state evaluation and demonstrate numerical agreement before claiming a benefit from the frozen sparse representation.